# FloodVerify: exact dataset counts (buildings and pixels per class, event and role)

**Runtime: CPU** (no GPU needed; High-RAM not needed). *Runtime > Run all*. About **20-30 min**, mostly copying the
data; the counting itself uses every CPU core (cell 0 prints how many).

**What it counts, straight from the source files, nothing sampled:**
1. **Coverage:** how many 1024 px images of each event exist in the xBD folders vs how many are in our manifest.
2. **Buildings:** every polygon in each post-disaster label file, once per 1024 px image (no double counting across
   crops), by damage class, including `un-classified`.
3. **Pixels:** every 512 px crop's damage mask, valid pixels only (the same rule as training and evaluation), plus
   no-data pixels.
4. **Cross-checks** against numbers we already quote (crop counts per role, Nepal 502 / 501 destroyed buildings,
   Florence 72,673 destroyed pixels, the 0.13% destroyed-pixel estimate).

Results are saved to `eval_extras/dataset_counts/` on Drive.

In [1]:
import os
NCPU = os.cpu_count(); print("CPUs:", NCPU)
from google.colab import drive
drive.mount("/content/drive")
D = "/content/drive/MyDrive/MAAI 902 X 901 PROJECT"
X, MAN = "/content/xbd", "/content/xbd/rscc_flood_manifest.csv"
OUT = f"{D}/eval_extras/dataset_counts"
os.makedirs(OUT, exist_ok=True)
%cd /content
!rm -rf /content/FloodVerify
!git clone -q https://github.com/aaarrvind/FloodVerify.git /content/FloodVerify
!pip install -q -e /content/FloodVerify --no-deps
import sys; sys.path.insert(0, "/content/FloodVerify/src"); sys.path.insert(0, "/content")

CPUs: 2
Mounted at /content/drive
/content
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for floodsitrep (pyproject.toml) ... done


In [2]:
# All four xBD folders: train and test (tars), hold and tier3 (Nepal) (10-15 min)
if not os.path.exists(f"{X}/tier3"):
    !mkdir -p /content/xbd
    !tar -xf "{D}/xbd_tars/train_images_labels_targets.tar" -C /content/xbd
    !tar -xf "{D}/xbd_tars/test_images_labels_targets.tar" -C /content/xbd
    !cp -r "{D}/xbd/hold" /content/xbd/
    !cp -r "{D}/xbd/tier3" /content/xbd/
    !cp "{D}/xbd/rscc_flood_manifest.csv" /content/xbd/
!ls /content/xbd

hold  rscc_flood_manifest.csv  test  tier3  train


In [3]:
%%writefile /content/dataset_counts_core.py
"""Exact dataset counts from the source files (label JSONs and masks). Owner: member 3.

Buildings: counted once per 1024 px image from the post-disaster label JSON (every polygon,
by its ``subtype``), so buildings crossing crop borders are not double counted.
Pixels: counted per 512 px crop from the damage mask, over valid pixels only (the same
``valid`` rule as training and evaluation), plus the number of no-data pixels.
"""
import json
from collections import Counter
from multiprocessing import Pool

import numpy as np

SUBTYPES = ("no-damage", "minor-damage", "major-damage", "destroyed", "un-classified")


def image_buildings(args):
    root, row = args
    from floodsitrep.data.labels_to_evidence import parse_wkt
    from floodsitrep.data.tiling import source_paths

    lab = json.load(open(source_paths(root, row)["label"]))
    c = Counter()
    for f in lab["features"]["xy"]:
        st = f["properties"].get("subtype", "missing")
        c[st] += 1
        if len(parse_wkt(f["wkt"])) < 3:
            c["_degenerate_polygon"] += 1
    return f"{row['event']}_{row['tile_id']}", row["event"], dict(c)


def crop_pixels(args):
    root, row = args
    from floodsitrep.data.tiling import load_crop, tile_name

    _, _, mask, valid = load_crop(root, row)
    counts = np.bincount(mask[valid].ravel(), minlength=256)
    other = int(counts[5:].sum())                      # any value outside 0..4 (should be 0)
    return tile_name(row), row["event"], counts[:5].tolist(), int((~valid).sum()), other


def run_pool(fn, root, rows, workers):
    with Pool(workers) as p:
        return p.map(fn, [(root, r) for r in rows], chunksize=16)

Writing /content/dataset_counts_core.py


In [4]:
import glob, json, numpy as np, pandas as pd
import dataset_counts_core as DC
from floodsitrep.data.tiling import read_manifest
from floodsitrep.data.splits import role_of
EVENTS = ["hurricane-harvey", "hurricane-matthew", "midwest-flooding", "hurricane-florence", "nepal-flooding"]
rows = read_manifest(MAN)
ROLE = {e: role_of(e) for e in EVENTS}
print("manifest rows (512 px crops):", len(rows))
print(pd.Series([r["event"] for r in rows]).value_counts().reindex(EVENTS).to_string())

# 1. Coverage: images on disk vs images in the manifest
cov = []
for e in EVENTS:
    on_disk = {}
    for sp in ("train", "test", "hold", "tier3"):
        n = len(glob.glob(f"{X}/{sp}/labels/{e}_*_post_disaster.json"))
        if n: on_disk[sp] = n
    imgs = {(r["split"], r["tile_id"]) for r in rows if r["event"] == e}
    parts = pd.Series([r["part"] for r in rows if r["event"] == e]).value_counts().to_dict()
    cov.append({"event": e, "role": ROLE[e], "images on disk": sum(on_disk.values()), "by xBD folder": on_disk,
                "images in manifest": len(imgs), "crops in manifest": sum(parts.values()),
                "crops per part 1/2/3/4": [parts.get(str(k), 0) for k in (1, 2, 3, 4)]})
COV = pd.DataFrame(cov)
print(COV.to_string(index=False))

manifest rows (512 px crops): 10148
hurricane-harvey      2088
hurricane-matthew     1620
midwest-flooding      1780
hurricane-florence    2184
nepal-flooding        2476
             event  role  images on disk                            by xBD folder  images in manifest  crops in manifest crops per part 1/2/3/4
  hurricane-harvey train             522  {'train': 319, 'test': 108, 'hold': 95}                 522               2088   [522, 522, 522, 522]
 hurricane-matthew train             405   {'train': 238, 'test': 73, 'hold': 94}                 405               1620   [405, 405, 405, 405]
  midwest-flooding train             445   {'train': 279, 'test': 80, 'hold': 86}                 445               1780   [445, 445, 445, 445]
hurricane-florence   val             546 {'train': 319, 'test': 108, 'hold': 119}                 546               2184   [546, 546, 546, 546]
    nepal-flooding  test             619                           {'tier3': 619}                 619        

In [5]:
# 2. Buildings, once per 1024 px image (every polygon in the post-disaster label file)
seen, img_rows = set(), []
for r in rows:
    k = (r["event"], r["split"], r["tile_id"])
    if k not in seen:
        seen.add(k); img_rows.append(r)
res = DC.run_pool(DC.image_buildings, X, img_rows, NCPU)
BI = pd.DataFrame([{"image": i, "event": e, **c} for i, e, c in res]).fillna(0)
for c in DC.SUBTYPES + ("_degenerate_polygon",):
    if c not in BI: BI[c] = 0
other = [c for c in BI.columns if c not in ("image", "event", "_degenerate_polygon") + DC.SUBTYPES]
print("unexpected subtypes:", other or "none")
BI["role"] = BI["event"].map(ROLE)
cls = ["no-damage", "minor-damage", "major-damage", "destroyed"]
BI["classified"] = BI[cls].sum(1)
by_event = BI.groupby("event")[cls + ["un-classified", "classified", "_degenerate_polygon"]].sum().reindex(EVENTS).fillna(0).astype(int)
by_event["images"] = BI.groupby("event").size().reindex(EVENTS).fillna(0).astype(int)
by_event["images with >=1 destroyed"] = (BI["destroyed"] > 0).groupby(BI["event"]).sum().reindex(EVENTS).fillna(0).astype(int)
by_role = BI.groupby("role")[cls + ["un-classified", "classified"]].sum().reindex(["train", "val", "test"]).fillna(0).astype(int)
by_role.loc["all"] = by_role.sum()
share = (100 * by_role[cls].div(by_role["classified"], axis=0)).round(2)
print("BUILDINGS per event"); print(by_event.to_string())
print("\nBUILDINGS per role"); print(by_role.to_string())
print("\n% of classified buildings per role"); print(share.to_string())

unexpected subtypes: none
BUILDINGS per event
                    no-damage  minor-damage  major-damage  destroyed  un-classified  classified  _degenerate_polygon  images  images with >=1 destroyed
event                                                                                                                                                  
hurricane-harvey        18638          4510         13378        848            581       37374                    0     522                        137
hurricane-matthew        4058         12331          2717       3524           1334       22630                    0     405                        280
midwest-flooding        12819           246           193        165            473       13423                    0     445                         63
hurricane-florence       8466           232          1949         81            820       10728                    0     546                         28
nepal-flooding          31225          513

In [6]:
# 3. Pixels per class, every crop, valid pixels only
res = DC.run_pool(DC.crop_pixels, X, rows, NCPU)
PX = pd.DataFrame([{"crop": t, "event": e, "bg": c[0], "no_damage": c[1], "minor": c[2], "major": c[3],
                    "destroyed": c[4], "no_data": nd, "other_values": o} for t, e, c, nd, o in res])
PX["role"] = PX["event"].map(ROLE)
pc = ["bg", "no_damage", "minor", "major", "destroyed"]
p_event = PX.groupby("event")[pc + ["no_data", "other_values"]].sum().reindex(EVENTS).fillna(0).astype(int)
p_role = PX.groupby("role")[pc + ["no_data", "other_values"]].sum().reindex(["train", "val", "test"]).fillna(0).astype(int)
p_role.loc["all"] = p_role.sum()
bld = ["no_damage", "minor", "major", "destroyed"]
print("PIXELS per event (valid pixels)"); print(p_event.to_string())
print("\nPIXELS per role"); print(p_role.to_string())
print("\n% of all valid pixels"); print((100 * p_role[pc].div(p_role[pc].sum(1), axis=0)).round(3).to_string())
print("\n% of building pixels"); print((100 * p_role[bld].div(p_role[bld].sum(1), axis=0)).round(2).to_string())
crops_d = (PX["destroyed"] > 0).groupby(PX["role"]).agg(["sum", "count"]).reindex(["train", "val", "test"])
crops_d.columns = ["crops with >=1 destroyed pixel", "crops"]
print("\n", crops_d.to_string())

PIXELS per event (valid pixels)
                           bg  no_damage    minor     major  destroyed   no_data  other_values
event                                                                                         
hurricane-harvey    450080070   36709646  8007920  20486588     735246  31337202             0
hurricane-matthew   391652662    1773394  6065315   1334093     843583  23004233             0
midwest-flooding    428557896   13828993   258418    226360     135081  23609572             0
hurricane-florence  541594774    9075656   594881   1804185      72673  19380327             0
nepal-flooding      592242546   14161662  2665276   2288721      89891  37620448             0

PIXELS per role
               bg  no_damage     minor     major  destroyed    no_data  other_values
role                                                                                
train  1270290628   52312033  14331653  22047041    1713910   77951007             0
val     541594774    9075656   

In [7]:
# 4. Cross-checks against numbers already quoted in our docs (should all say OK, or explain why not)
def check(name, got, expected, tol=0):
    ok = abs(got - expected) <= tol
    print(f"{'OK  ' if ok else 'DIFF'} {name}: got {got:,} expected {expected:,}")
n_role = PX["role"].value_counts()
check("train crops", int(n_role["train"]), 5488); check("val crops", int(n_role["val"]), 2184)
check("test crops", int(n_role["test"]), 2476)
check("Nepal destroyed buildings (data README)", int(by_event.loc["nepal-flooding", "destroyed"]), 502)
print("     (our building-level evaluation scored 501; it skips buildings with no valid pixels)")
check("Florence destroyed pixels (destroyed notebook, 19 Nov)", int(p_event.loc["hurricane-florence", "destroyed"]), 72673)
check("mask values outside 0-4", int(PX["other_values"].sum()), 0)
d_share = 100 * p_role.loc["train", "destroyed"] / p_role.loc["train", pc].sum()
print(f"     train destroyed share of valid pixels: {d_share:.3f}% (README estimate from 500 sampled tiles: 0.13%)")

OK   train crops: got 5,488 expected 5,488
OK   val crops: got 2,184 expected 2,184
OK   test crops: got 2,476 expected 2,476
OK   Nepal destroyed buildings (data README): got 502 expected 502
     (our building-level evaluation scored 501; it skips buildings with no valid pixels)
OK   Florence destroyed pixels (destroyed notebook, 19 Nov): got 72,673 expected 72,673
OK   mask values outside 0-4: got 0 expected 0
     train destroyed share of valid pixels: 0.126% (README estimate from 500 sampled tiles: 0.13%)


In [8]:
# 5. Save
COV.to_csv(f"{OUT}/coverage.csv", index=False)
by_event.to_csv(f"{OUT}/buildings_by_event.csv"); by_role.to_csv(f"{OUT}/buildings_by_role.csv")
p_event.to_csv(f"{OUT}/pixels_by_event.csv"); p_role.to_csv(f"{OUT}/pixels_by_role.csv")
BI.to_csv(f"{OUT}/buildings_per_image.csv", index=False); PX.to_csv(f"{OUT}/pixels_per_crop.csv", index=False)
print("saved to", OUT); print(sorted(os.listdir(OUT)))

saved to /content/drive/MyDrive/MAAI 902 X 901 PROJECT/eval_extras/dataset_counts
['buildings_by_event.csv', 'buildings_by_role.csv', 'buildings_per_image.csv', 'coverage.csv', 'pixels_by_event.csv', 'pixels_by_role.csv', 'pixels_per_crop.csv']


Paste every printed output in the chat (sections 1-4). Then *Runtime > Disconnect and delete runtime*.